In [37]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from config import CAT_VARS, DATA, NUM_VARS
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from lib.model_development import LightGbmModelBuilder

In [38]:
dataset_path = DATA / "processed" / "basic_data_prep"
datasets = {f.stem: pd.read_csv(f) for f in sorted(dataset_path.glob("*.csv"))}
for k, v in datasets.items():
    print(f"{k}: \n{v}\n")

X_test: 
      X01_LIMIT_BAL  X02_SEX  X03_EDUCATION  X04_MARRIAGE  X05_AGE  X06_PAY_0  \
0             50000        1              2             2       46         -1   
1            150000        1              1             1       31         -1   
2             50000        1              2             2       25          0   
3            290000        2              1             2       25          0   
4            500000        2              2             1       27         -2   
...             ...      ...            ...           ...      ...        ...   
5995         150000        2              4             2       27         -2   
5996          50000        1              1             2       24          2   
5997         220000        1              1             2       34          0   
5998         120000        1              1             2       26         -1   
5999         200000        1              3             2       33         -2   

      X07_PAY_2  X

In [39]:
model_params_dict = {'verbose': 2}
results_dict = lgbm_model = LightGbmModelBuilder.from_split_data(datasets['X_train'], datasets['y_train'], datasets['X_test'],datasets['y_test'], model_params_dict).run()

[LightGBM] [Info] Number of positive: 5309, number of negative: 18691
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.218929
[LightGBM] [Debug] init for col-wise cost 0.000002 seconds, init for row-wise cost 0.000818 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total Bins 3263
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.221208 -> initscore=-1.258639
[LightGBM] [Info] Start training from score -1.258639
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Traine

# Validate results

In [40]:
y_score = results_dict['pred_test']
y_true = results_dict['y_test']
print(type(y_true))
print(type(y_score))
print(y_true)
print(y_score)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
[0 0 0 ... 0 0 0]
[0.1677617  0.11013097 0.15932789 ... 0.02682496 0.15884646 0.06113354]


In [41]:
auc = roc_auc_score(y_true, y_score)
gini = 2 * auc - 1

metrics = {"auc": auc, "gini": gini}

print(metrics)

{'auc': 0.7763053349977771, 'gini': 0.5526106699955542}


In [42]:
fpr, tpr, thr = roc_curve(y_true, y_score)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        hoverinfo="skip",
        line={"color": "#B0B4BA", "width": 1, "dash": "dash"},
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        customdata=thr,
        line={"color": "#2B6CB0", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"ROC — logistic champion, test set (AUC {auc:.3f}, Gini {2 * auc - 1:.3f})",
    xaxis={
        "title": "False positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
    },
    yaxis={
        "title": "True positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
        "scaleanchor": "x",
        "scaleratio": 1,
    },
    template="plotly_white",
    width=560,
    height=560,
    margin={"l": 60, "r": 30, "t": 60, "b": 60},
)
fig.show()

**Conclusion:** OK, so Gini is working well for the lowest scores but stops working quite as well around the middle range. Can try to get a better performing model to tell whether this is a limitation of the dataset or the model.